In [1]:
"""
Defect-Geometric Completion Prototype

Toy problem:
- Defects are graph nodes.
- Connections are U(1) phases phi_ij encoding relational transport.
- Curvature around a triangle is phi_ij + phi_jk + phi_ki.
- Completion infers missing relations by minimizing:
    data fidelity + lambda_loop * loop curvature.

Run:
    python dg_completion_prototype.py
"""

import math
import random
import numpy as np
import torch


def wrap_angle(x):
    return torch.atan2(torch.sin(x), torch.cos(x))


def circ_error(a, b):
    return wrap_angle(a - b)


def make_problem(n=6, obs_prob=0.48, noise=0.04, seed=7):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    theta_true = torch.rand(n) * 2 * math.pi - math.pi
    phi_true = torch.zeros(n, n)

    for i in range(n):
        for j in range(n):
            phi_true[i, j] = wrap_angle(theta_true[j] - theta_true[i])

    mask = torch.zeros(n, n, dtype=torch.bool)
    phi_obs = torch.zeros(n, n)

    for i in range(n):
        for j in range(i + 1, n):
            if random.random() < obs_prob:
                val = wrap_angle(phi_true[i, j] + noise * torch.randn(()))
                mask[i, j] = mask[j, i] = True
                phi_obs[i, j] = val
                phi_obs[j, i] = -val

    return theta_true, phi_true, mask, phi_obs


def mean_triangle_curvature(phi):
    n = phi.shape[0]
    vals = []
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                vals.append(abs(float(wrap_angle(phi[i, j] + phi[j, k] + phi[k, i]))))
    return sum(vals) / len(vals)


def complete_connections(phi_obs, mask, steps=2000, lr=0.06, lambda_loop=2.0, seed=11):
    torch.manual_seed(seed)
    n = phi_obs.shape[0]

    raw = torch.nn.Parameter(0.2 * torch.randn(n, n))
    opt = torch.optim.Adam([raw], lr=lr)
    tri_idx = [(i, j, k) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)]
    zero = torch.tensor(0.0)
    hist = []

    for t in range(steps):
        opt.zero_grad()

        # Antisymmetrized U(1) connection: phi_ji = -phi_ij
        phi = wrap_angle(raw - raw.T)

        data_loss = (circ_error(phi[mask], phi_obs[mask]) ** 2).mean()
        loop_loss = torch.stack([
            circ_error(phi[i, j] + phi[j, k] + phi[k, i], zero) ** 2
            for i, j, k in tri_idx
        ]).mean()

        loss = data_loss + lambda_loop * loop_loss
        loss.backward()
        opt.step()

        if t % 200 == 0 or t == steps - 1:
            hist.append((t, loss.detach().item(), data_loss.detach().item(), loop_loss.detach().item()))

    with torch.no_grad():
        phi = wrap_angle(raw - raw.T).detach()

    return phi, hist


def evaluate(phi_pred, phi_true, mask):
    n = phi_true.shape[0]
    all_pairs = torch.ones(n, n, dtype=torch.bool)
    all_pairs.fill_diagonal_(False)

    missing = all_pairs & (~mask)
    observed = all_pairs & mask

    return {
        "observed_mae_rad": float(circ_error(phi_pred[observed], phi_true[observed]).abs().mean()),
        "missing_mae_rad": float(circ_error(phi_pred[missing], phi_true[missing]).abs().mean()),
        "all_mae_rad": float(circ_error(phi_pred[all_pairs], phi_true[all_pairs]).abs().mean()),
        "mean_triangle_curvature_rad": mean_triangle_curvature(phi_pred),
    }


def main():
    # Pick the same deterministic example used in the chat.
    theta_true, phi_true, mask, phi_obs = make_problem(seed=7)

    phi_pred, hist = complete_connections(phi_obs, mask)

    baseline = torch.zeros_like(phi_true)
    baseline[mask] = phi_obs[mask]

    print("observed directed edges:", int(mask.sum().item()))
    print("missing directed edges:", int((~mask).sum().item() - phi_true.shape[0]))
    print("final training record:", hist[-1])
    print("\nDG completion metrics:")
    print(evaluate(phi_pred, phi_true, mask))
    print("\nZero-fill baseline metrics:")
    print(evaluate(baseline, phi_true, mask))

    print("\nMissing upper-triangle edges: i, j, true, predicted, abs_error_rad")
    for i in range(phi_true.shape[0]):
        for j in range(i + 1, phi_true.shape[0]):
            if not mask[i, j]:
                err = abs(float(circ_error(phi_pred[i, j], phi_true[i, j])))
                print(i, j, float(phi_true[i, j]), float(phi_pred[i, j]), err)


if __name__ == "__main__":
    main()


observed directed edges: 22
missing directed edges: 8
final training record: (1999, 0.0006002733716741204, 0.0004316940903663635, 8.428964792983606e-05)

DG completion metrics:
{'observed_mae_rad': 0.023959755897521973, 'missing_mae_rad': 0.03334078937768936, 'all_mae_rad': 0.026461362838745117, 'mean_triangle_curvature_rad': 0.004750329325906933}

Zero-fill baseline metrics:
{'observed_mae_rad': 0.029647307470440865, 'missing_mae_rad': 0.7816534638404846, 'all_mae_rad': 0.23018230497837067, 'mean_triangle_curvature_rad': 0.47748889960348606}

Missing upper-triangle edges: i, j, true, predicted, abs_error_rad
0 3 0.7663456797599792 0.7063266634941101 0.06001901626586914
0 5 -0.6902778148651123 -0.7167354822158813 0.026457667350769043
1 4 0.21336686611175537 0.19454607367515564 0.018820790573954582
3 5 -1.4566235542297363 -1.4285578727722168 0.02806568145751953
